<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-03/AI_Modul_3.2_Praktikum_Instalasi_dan_Environment.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 3.2: Praktikum Instalasi dan Environment
### Manajemen Instalasi Python, Isolasi Lingkungan Virtual (venv & Conda), Konfigurasi IDE VS Code, Ekosistem Kernel Jupyter, dan Reproduksibilitas Dependensi Proyek AI Agribisnis

---

> **Diktat Terkait:** [AI_Modul_3.2_Instalasi_dan_Environment.md](../../docs/part-03/AI_Modul_3.2_Instalasi_dan_Environment.md)  
> **Outputs:** Skrip audit isolasi lingkungan virtual, validator berkas manifest `requirements.txt`, serta manajer snapshot dependensi proyek machine learning perkebunan kelapa sawit.  
> **Outcomes:** Mahasiswa terampil mengisolasi ruang dependensi Python, mencegah tabrakan versi pustaka (*Dependency Hell*), membedakan mekanisme kerja `venv` dan `conda`, serta menguasai alur kerja *reproducible AI*.  
> **Impacts:** Terjaminnya stabilitas sistem inferensi cerdas saat dialihkan dari komputer riset ke server produksi pabrik atau perangkat tepi (*Edge AI Gateway*) di perkebunan.
>
> **Tujuan Praktikum:**  
> 1. Mendeteksi status isolasi interpreter CPython aktif secara programatis via `sys.prefix`.  
> 2. Mengaudit keberadaan dan versi paket pihak ketiga menggunakan `importlib.metadata`.  
> 3. Membangun parser berkas manifest dependensi `requirements.txt`.  
> 4. Menjalankan pipeline diagnostik industri `AuditorEnvironmentAI`.  
> 5. Menyelesaikan 3 tantangan: verifikator status venv, validator manifest dependensi, dan manajer snapshot versi pustaka.

## 1. Persiapan Environment & Pemuatan Library
Jalankan sel berikut untuk memuat pustaka standar inspeksi sistem dan pustaka visualisasi:

In [ ]:
import sys
import os
import platform
from pathlib import Path
import importlib.metadata
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

# Modul numerik & visualisasi
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Konfigurasi visualisasi grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("[OK] Seluruh pustaka dasar dan analitika berhasil diinisialisasi.")
print(f"[INFO] Executable Interpreter : {sys.executable}")
print(f"[INFO] Platform Sistem        : {platform.platform()}")

## 2. Deteksi Programatis Status Isolasi Virtual Environment
Pada interpreter CPython resmi, isolasi virtual environment diverifikasi melalui komparasi antara variabel `sys.prefix` (direktori eksekusi lokal) dan `sys.base_prefix` (instalasi induk host).

In [ ]:
# Pemeriksaan variabel penunjuk jalur CPython
jalur_prefix = sys.prefix
jalur_base_prefix = sys.base_prefix
is_virtual_env = jalur_prefix != jalur_base_prefix

# Deteksi Conda Environment
is_conda = "CONDA_PREFIX" in os.environ
nama_conda_env = os.environ.get("CONDA_DEFAULT_ENV", "-")

print("=" * 75)
print("HASIL INSPEKSI ISOLASI RUNTIME INTERPRETER")
print("=" * 75)
print(f"Prefix Aktif (sys.prefix)          : {jalur_prefix}")
print(f"Base Prefix Host (sys.base_prefix) : {jalur_base_prefix}")
print(f"Status Lingkungan Terisolasi (venv): {is_virtual_env}")
print(f"Terdeteksi Conda Environment       : {is_conda} (Nama: {nama_conda_env})")

if is_virtual_env or is_conda:
    print("\n[STATUS AMAN] Interpreter berjalan di dalam ruang terisolasi mandiri.")
else:
    print("\n[PERINGATAN] Program berjalan pada interpreter sistem global!")
    print("Disarankan mengaktifkan virtual environment (.venv) untuk mencegah Dependency Hell.")
print("=" * 75)

## 3. Audit Dependensi Paket Eksternal Menggunakan `importlib.metadata`
Mengevaluasi ketersediaan pustaka-pustaka inti sains data dan AI agribisnis tanpa perlu mengeksekusi impor kode berat di memori.

In [ ]:
# Daftar paket standar analisis kecerdasan buatan perkebunan
target_paket_ai = [
    "numpy", "pandas", "matplotlib", "scipy", "scikit-learn",
    "opencv-python", "torch", "ipykernel"
]

status_audit = []
print("=== STATUS AUDIT PUSTAKA KECERDASAN BUATAN AGRIBISNIS ===")
print(f"{'Nama Paket':<18} | {'Status':<15} | {'Versi Terdeteksi'}")
print("-" * 55)

for paket in target_paket_ai:
    try:
        ver = importlib.metadata.version(paket)
        status_audit.append((paket, True, ver))
        print(f"{paket:<18} | {'[V] Terpasang':<15} | {ver}")
    except importlib.metadata.PackageNotFoundError:
        status_audit.append((paket, False, "Belum Terpasang"))
        print(f"{paket:<18} | {'[X] Hilang':<15} | -")

# Visualisasi Status Ketersediaan Dependensi
nama_pkg = [item[0] for item in status_audit]
terpasang_flag = [1 if item[1] else 0 for item in status_audit]
warna_bar = ['forestgreen' if f == 1 else 'indianred' for f in terpasang_flag]

fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.barh(nama_pkg, terpasang_flag, color=warna_bar, edgecolor='black', height=0.6)
ax.set_xlim(0, 1.2)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Belum Terpasang (0)', 'Terpasang Siap Pakai (1)'], fontweight='bold')
ax.set_title('Kesiapan Pustaka Inti Proyek AI Perkebunan', fontweight='bold', fontsize=11)

for i, bar in enumerate(bars):
    label_teks = status_audit[i][2]
    ax.text(0.05, bar.get_y() + bar.get_height()/2., f"Status: {label_teks}", 
            va='center', color='white' if status_audit[i][1] else 'black', fontweight='bold', fontsize=8.5)

plt.tight_layout()
plt.savefig('docs/assets/status_dependensi_proyek_ai.png', dpi=150)
print("\n[INFO] Grafik status dependensi berhasil disimpan ke 'docs/assets/status_dependensi_proyek_ai.png'.")

## 4. Parser & Validator Berkas Manifest `requirements.txt`
Membaca berkas teks manifest dependensi, memisahkan nama paket dan operator versi semantik (`==`), lalu memverifikasi kesesuaiannya dengan pustaka aktif.

In [ ]:
# Contoh manifest dependensi proyek kebun sawit
simulasi_manifest_teks = """# Dependensi Proyek AI Kebun Sawit
numpy==1.26.4
pandas>=2.0.0
matplotlib==3.8.4
# Pustaka khusus drone
opencv-python==4.8.0
scipy==1.13.0
"""

jalur_manifest_uji = Path("requirements_test_sawit.txt")
jalur_manifest_uji.write_text(simulasi_manifest_teks, encoding="utf-8")

def validasi_manifest_requirements(jalur_file: str) -> Dict[str, Any]:
    """
    Membaca dan memvalidasi integritas pustaka yang tercantum pada requirements.txt.
    """
    path = Path(jalur_file)
    if not path.exists():
        return {"sukses": False, "pesan": f"Berkas {jalur_file} tidak ditemukan."}

    paket_valid = []
    paket_hilang = []
    paket_mismatch = []

    with path.open("r", encoding="utf-8") as f:
        for baris in f:
            baris = baris.strip()
            if not baris or baris.startswith("#"):
                continue

            # Parsing operator == atau >=
            if "==" in baris:
                nama, versi_diminta = baris.split("==", 1)
                op = "=="
            elif ">=" in baris:
                nama, versi_diminta = baris.split(">=", 1)
                op = ">="
            else:
                nama, versi_diminta, op = baris, "-", "any"

            nama = nama.strip()
            versi_diminta = versi_diminta.strip()

            try:
                versi_aktif = importlib.metadata.version(nama)
                if op == "==" and versi_aktif != versi_diminta:
                    paket_mismatch.append((nama, versi_diminta, versi_aktif))
                else:
                    paket_valid.append((nama, versi_aktif))
            except importlib.metadata.PackageNotFoundError:
                paket_hilang.append((nama, versi_diminta))

    return {
        "sukses": True,
        "paket_valid": paket_valid,
        "paket_mismatch": paket_mismatch,
        "paket_hilang": paket_hilang,
        "status_reproduksi_penuh": len(paket_hilang) == 0 and len(paket_mismatch) == 0
    }

laporan_validasi = validasi_manifest_requirements(str(jalur_manifest_uji))

print("=== HASIL VALIDASI MANIFEST REQUIREMENTS.TXT ===")
print(f"Status Reproduksi Penuh : {laporan_validasi['status_reproduksi_penuh']}")
print(f"Paket Terpasang Cocok   : {laporan_validasi['paket_valid']}")
print(f"Paket Versi Tidak Cocok : {laporan_validasi['paket_mismatch']}")
print(f"Paket Belum Terpasang   : {laporan_validasi['paket_hilang']}")

# Bersihkan berkas uji
jalur_manifest_uji.unlink()

## 5. Implementasi Terpadu Kelas Industri: `AuditorEnvironmentAI`
Menggabungkan modul audit isolasi dan audit paket ke dalam satu objek transfer data (@dataclass) yang terstruktur dan aman.

In [ ]:
@dataclass(frozen=True)
class StatusVirtualEnv:
    is_virtual_env: bool
    jalur_prefix: str
    jalur_base_prefix: str
    nama_env: str

@dataclass(frozen=True)
class LaporanPaket:
    nama_paket: str
    terpasang: bool
    versi_terdeteksi: Optional[str]
    status_kepatuhan: str

class AuditorEnvironmentAI:
    """
    Auditor kepatuhan lingkungan komputasi untuk menjamin stabilitas
    dan reproduksibilitas pipeline model kecerdasan buatan perkebunan.
    """
    @staticmethod
    def deteksi_virtual_environment() -> StatusVirtualEnv:
        is_venv = sys.prefix != sys.base_prefix
        nama = Path(sys.prefix).name if is_venv else "GLOBAL_SYSTEM"
        return StatusVirtualEnv(
            is_virtual_env=is_venv,
            jalur_prefix=sys.prefix,
            jalur_base_prefix=sys.base_prefix,
            nama_env=nama
        )

    @staticmethod
    def audit_daftar_dependensi(daftar_kebutuhan: Dict[str, str]) -> List[LaporanPaket]:
        laporan: List[LaporanPaket] = []
        for nama_paket, _ in daftar_kebutuhan.items():
            try:
                versi_aktif = importlib.metadata.version(nama_paket)
                laporan.append(LaporanPaket(
                    nama_paket=nama_paket,
                    terpasang=True,
                    versi_terdeteksi=versi_aktif,
                    status_kepatuhan=f"Terpasang (v{versi_aktif})"
                ))
            except importlib.metadata.PackageNotFoundError:
                laporan.append(LaporanPaket(
                    nama_paket=nama_paket,
                    terpasang=False,
                    versi_terdeteksi=None,
                    status_kepatuhan="[GALAT] Belum Terpasang!"
                ))
        return laporan

# Eksekusi Audit
auditor = AuditorEnvironmentAI()
status_env = auditor.deteksi_virtual_environment()
daftar_audit = {"numpy": "1.24.0", "pandas": "2.0.0", "matplotlib": "3.7.0", "scipy": "1.10.0"}
hasil_pkg = auditor.audit_daftar_dependensi(daftar_audit)

print("=" * 75)
print("LAPORAN INTEGRITAS LINGKUNGAN AI PERKEBUNAN INSTIPER")
print("=" * 75)
print(f"Status Virtualenv : {status_env.is_virtual_env} ({status_env.nama_env})")
print(f"Jalur Prefix     : {status_env.jalur_prefix}")
print("\nStatus Paket Terverifikasi:")
for item in hasil_pkg:
    print(f" * {item.nama_paket:<15} : {item.status_kepatuhan}")
print("=" * 75)

## 6. Solusi Lengkap Tantangan Pemrograman Berjenjang
Berikut adalah implementasi komprehensif dari ketiga tantangan mandiri yang terdapat pada diktat perkuliahan:

### Tantangan 1 (Tingkat Dasar): Pendeteksi Otomatis Status Virtual Environment
Mengecek status isolasi dan memberikan instruksi perintah terminal aktivasi yang tepat sesuai sistem operasi aktif.

In [ ]:
def verifikasi_status_virtualenv() -> Dict[str, Any]:
    """
    Memvalidasi isolasi environment dan menyediakan instruksi setup terminal yang tepat.
    """
    is_venv = sys.prefix != sys.base_prefix
    os_name = sys.platform
    
    if is_venv:
        rekomendasi = "Lingkungan virtual aktif dan aman untuk instalasi dependensi."
    else:
        if os_name == "win32":
            rekomendasi = "Jalankan: python -m venv .venv lalu .venv\\Scripts\\Activate.ps1"
        else:
            rekomendasi = "Jalankan: python3 -m venv .venv lalu source .venv/bin/activate"
            
    return {
        "terisolasi": is_venv,
        "platform": os_name,
        "direktori_prefix": sys.prefix,
        "rekomendasi_tindakan": rekomendasi
    }

# Pengujian Tantangan 1
hasil_t1 = verifikasi_status_virtualenv()
print("=== TANTANGAN 1: STATUS VIRTUAL ENVIRONMENT ===")
print(f"Terisolasi           : {hasil_t1['terisolasi']}")
print(f"Platform Terdeteksi  : {hasil_t1['platform']}")
print(f"Rekomendasi Tindakan : {hasil_t1['rekomendasi_tindakan']}")

### Tantangan 2 (Tingkat Menengah): Parser & Validator Manifest `requirements.txt`
Memvalidasi manifest requirements proyek kebun terhadap pustaka yang aktif di memori.

In [ ]:
# Kita panggil fungsi yang telah dirancang pada Section 4
manifest_demo = Path("requirements_sawit_demo.txt")
manifest_demo.write_text("""# Requirements Resmi Deteksi Ganoderma
numpy==1.26.4
pandas>=2.0.0
matplotlib>=3.5.0
scipy>=1.9.0
""", encoding="utf-8")

hasil_t2 = validasi_manifest_requirements(str(manifest_demo))
manifest_demo.unlink()

print("=== TANTANGAN 2: HASIL VALIDASI MANIFEST SAWIT ===")
print(f"Validasi Sukses : {hasil_t2['sukses']}")
print(f"Reproduksi Penuh: {hasil_t2['status_reproduksi_penuh']}")
print(f"Daftar Paket Valid:")
for p, v in hasil_t2["paket_valid"]:
    print(f" -> {p:<15} (Versi Terpasang: {v})")

### Tantangan 3 (Tingkat Mahir): Manajer Snapshot & Penjaga Kompatibilitas Versi Dependensi
Membangun kelas `SnapshotEnvironmentManager` yang mengambil snapshot versi pustaka, membandingkan dua snapshot berbeda untuk mendeteksi perubahan, dan mengekspor manifest terkunci.

In [ ]:
class SnapshotEnvironmentManager:
    """
    Manajer pelacak snapshot versi pustaka untuk mendeteksi anomali pembaruan dependensi.
    """
    @staticmethod
    def ambil_snapshot_aktif(daftar_paket: List[str]) -> Dict[str, str]:
        """Merekam snapshot versi dari paket yang terpasang."""
        snapshot: Dict[str, str] = {}
        for pkg in daftar_paket:
            try:
                snapshot[pkg] = importlib.metadata.version(pkg)
            except importlib.metadata.PackageNotFoundError:
                snapshot[pkg] = "TIDAK_TERPASANG"
        return snapshot

    @staticmethod
    def bandingkan_snapshot(snapshot_lama: Dict[str, str], snapshot_baru: Dict[str, str]) -> Dict[str, Any]:
        """Membandingkan dua snapshot untuk mendeteksi upgrade, downgrade, atau paket baru."""
        perubahan = []
        seluruh_kunci = set(snapshot_lama.keys()) | set(snapshot_baru.keys())
        
        for k in sorted(list(seluruh_kunci)):
            v_lama = snapshot_lama.get(k, "BELUM_ADA")
            v_baru = snapshot_baru.get(k, "DIHAPUS")
            if v_lama != v_baru:
                perubahan.append({"paket": k, "lama": v_lama, "baru": v_baru})
                
        return {
            "total_perubahan": len(perubahan),
            "detail_perubahan": perubahan
        }

    @staticmethod
    def simpan_manifest_terkunci(jalur_output: str, snapshot: Dict[str, str]) -> None:
        """Mengekspor snapshot ke berkas teks requirements terkunci."""
        konten = [f"{pkg}=={ver}" for pkg, ver in snapshot.items() if ver != "TIDAK_TERPASANG"]
        Path(jalur_output).write_text("\n".join(konten) + "\n", encoding="utf-8")

# Pengujian Tantangan 3
koleksi_uji = ["numpy", "pandas", "matplotlib", "scipy"]
snapshot_sekarang = SnapshotEnvironmentManager.ambil_snapshot_aktif(koleksi_uji)

# Simulasi snapshot lama (misal 6 bulan lalu)
snapshot_lama_simulasi = {
    "numpy": "1.24.0",
    "pandas": "2.0.0",
    "matplotlib": snapshot_sekarang.get("matplotlib", "3.7.0"),
    "scipy": "1.9.0"
}

hasil_komparasi = SnapshotEnvironmentManager.bandingkan_snapshot(snapshot_lama_simulasi, snapshot_sekarang)

print("=== TANTANGAN 3: AUDIT PERUBAHAN DEPENDENSI ANTAR-SNAPSHOT ===")
print(f"Total Dependensi Berubah Versi: {hasil_komparasi['total_perubahan']}")
for diff in hasil_komparasi["detail_perubahan"]:
    print(f" -> Pustaka {diff['paket']:<12}: Versi Lama [{diff['lama']}] ==> Versi Baru [{diff['baru']}]")